In [1]:
from collections import Counter

training_words = [
    "happy",
    "happier",
    "happiness",
    "help",
    "helper",
    "helping"
]

vocabulary = {}

for word in training_words:
    tokenized = " ".join(list(word)) + " <eow>"
    vocabulary[tokenized] = vocabulary.get(tokenized, 0) + 1

def calculate_pairs(vocab):
    pair_counter = Counter()

    for word, frequency in vocab.items():
        symbols = word.split()

        for index in range(len(symbols) - 1):
            pair_counter[(symbols[index], symbols[index + 1])] += frequency

    return pair_counter

def combine_pair(selected_pair, vocab):
    updated_vocab = {}

    old_pattern = " ".join(selected_pair)
    new_pattern = "".join(selected_pair)

    for word in vocab:
        merged_word = word.replace(old_pattern, new_pattern)
        updated_vocab[merged_word] = vocab[word]

    return updated_vocab

merge_history = []
merge_iterations = 8

for step in range(merge_iterations):

    pair_stats = calculate_pairs(vocabulary)

    if len(pair_stats) == 0:
        break

    frequent_pair = max(pair_stats, key=pair_stats.get)

    merge_history.append(frequent_pair)

    vocabulary = combine_pair(frequent_pair, vocabulary)

def bpe_encode(text, merge_rules):

    pieces = list(text) + ["<eow>"]

    for rule in merge_rules:

        position = 0

        while position < len(pieces) - 1:

            if pieces[position] == rule[0] and pieces[position + 1] == rule[1]:

                pieces = pieces[:position] + [rule[0] + rule[1]] + pieces[position + 2:]

            else:
                position += 1

    return pieces

def bpe_decode(tokens):
    return "".join(tokens).replace("<eow>", "")

sample_word = "helper"

encoded_output = bpe_encode(sample_word, merge_history)
decoded_output = bpe_decode(encoded_output)

print("Learned Merge Rules:")
for i, rule in enumerate(merge_history, 1):
    print(i, ":", rule)

print("\nOriginal Word :", sample_word)
print("Encoded Tokens :", encoded_output)
print("Decoded Word :", decoded_output)

print("\nFinal Vocabulary:")
for item in sorted(vocabulary.keys()):
    print(item)

Learned Merge Rules:
1 : ('h', 'a')
2 : ('ha', 'p')
3 : ('hap', 'p')
4 : ('h', 'e')
5 : ('he', 'l')
6 : ('hel', 'p')
7 : ('happ', 'i')
8 : ('e', 'r')

Original Word : helper
Encoded Tokens : ['help', 'er', '<eow>']
Decoded Word : helper

Final Vocabulary:
happ y <eow>
happi er <eow>
happi n e s s <eow>
help <eow>
help er <eow>
help i n g <eow>
